# Ⅰ第6回 演習1 転移学習の復習 ― 最終層を置き換えて，そこだけを学習する

学習済みモデルを自分の分類に使う手順は，どのモデルでも同じ 3 つ．

1. 学習済みモデルをダウンロードする
2. 最終層の **名前** と **出力数** を確かめる（ImageNet 用なので出力は 1000）
3. 全部を凍結し，最終層を分けたいクラスの数の出力に置き換える（新しい層だけが学習する）

- `___` の所を書き換える．`___` が残っていると `NameError: name '___' is not defined` が出る
- データは使わない．LLM は使わない

## (0) 準備

In [1]:
# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from common.tlcheck import check_model
print("準備 OK")

準備 OK


![ResNet18 の層ごとの学習する（黄）・凍結（灰）](img/freeze.png)

## 問題1 ResNet18 を 10 クラスに

### (1-1) 学習済みモデルをダウンロードする（そのまま実行）

初回はダウンロードする（45MB．第5回に済んでいれば不要）．`print(model)` で層の一覧が出る．**一番下が最終層**．

In [2]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)   # 学習済みの重みを読む
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

### (1-2) 穴埋め：最終層の名前と出力数を確かめる

**問い**：(1-1) の一覧の一番下を見て，`___` を最終層の名前に書き換えて実行しなさい．

In [3]:
print(model.fc)                                   # 空欄（3 か所）に最終層の名前を書く
print("入る数:", model.fc.in_features, "／ 出力数:", model.fc.out_features)

Linear(in_features=512, out_features=1000, bias=True)
入る数: 512 ／ 出力数: 1000


<details><summary>答え（考えてから開く）</summary>

```python
print(model.fc)                                   # 空欄（3 か所）に最終層の名前を書く
print("入る数:", model.fc.in_features, "／ 出力数:", model.fc.out_features)
```

</details>

このセルをダブルクリックして書く．最終層の名前：fc　入る数：512　出力数：1000（ImageNet の何種類か）

### (1-3) 穴埋め：出力を 10 個に変え，最終層だけを学習させる

**問い**：`___` を埋めて，10 クラスに分類し，最終層だけを学習する `model` にしなさい．実行したら下の確認セルを実行する．`OK` が出れば正解（ちがうと直し方が出る）．

In [4]:
NUM_CLASSES = 10
for param in model.parameters():                   # 1 全部を凍結する（読んだ直後は全部が学習する設定）
    param.requires_grad = False                     #   True か False
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)  # 2 最終層を NUM_CLASSES 個の出力に置き換える（新しい層は学習する）

<details><summary>答え（考えてから開く）</summary>

```python
NUM_CLASSES = 10
for param in model.parameters():                   # 1 全部を凍結する（読んだ直後は全部が学習する設定）
    param.requires_grad = False                      #   True か False
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)  # 2 最終層を NUM_CLASSES 個の出力に置き換える（新しい層は学習する）
```

</details>

In [5]:
check_model(model, 10, "last")

OK｜出力 10 個｜学習するパラメータ 5,130 / 11,181,642（0.05%）


## 問題2 DenseNet121 を 3 クラスに（最終層の名前が違うモデル）

### (2-1) 学習済みモデルをダウンロードする（そのまま実行）

初回はダウンロードする（31MB）．`print(model)` で層の一覧が出る．**一番下が最終層**．

In [6]:
model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)   # 学習済みの重みを読む
print(model)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /Users/konnotakuto/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


  0%|          | 0.00/30.8M [00:00<?, ?B/s]

  6%|▌         | 1.88M/30.8M [00:00<00:01, 19.6MB/s]

 14%|█▍        | 4.38M/30.8M [00:00<00:01, 23.4MB/s]

 21%|██▏       | 6.62M/30.8M [00:00<00:01, 18.6MB/s]

 31%|███       | 9.50M/30.8M [00:00<00:01, 21.1MB/s]

... (7 outputs left out)

 86%|████████▋ | 26.6M/30.8M [00:01<00:00, 19.0MB/s]

 93%|█████████▎| 28.6M/30.8M [00:01<00:00, 15.7MB/s]

100%|██████████| 30.8M/30.8M [00:02<00:00, 16.1MB/s]

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, bias=T

### (2-2) 穴埋め：最終層の名前と出力数を確かめる

**問い**：(2-1) の一覧の一番下を見て，`___` を最終層の名前に書き換えて実行しなさい．

In [7]:
print(model.classifier)                                   # 空欄（3 か所）に最終層の名前を書く
print("入る数:", model.classifier.in_features, "／ 出力数:", model.classifier.out_features)

Linear(in_features=1024, out_features=1000, bias=True)
入る数: 1024 ／ 出力数: 1000


<details><summary>答え（考えてから開く）</summary>

```python
print(model.classifier)                                   # 空欄（3 か所）に最終層の名前を書く
print("入る数:", model.classifier.in_features, "／ 出力数:", model.classifier.out_features)
```

</details>

このセルをダブルクリックして書く．最終層の名前：classifier　入る数：1024　出力数：1000（ImageNet の何種類か）

### (2-3) 穴埋め：出力を 3 個に変え，最終層だけを学習させる

**問い**：`___` を埋めて，3 クラスに分類し，最終層だけを学習する `model` にしなさい．実行したら下の確認セルを実行する．`OK` が出れば正解（ちがうと直し方が出る）．

In [8]:
NUM_CLASSES = 3
for param in model.parameters():                   # 1 全部を凍結する（読んだ直後は全部が学習する設定）
    param.requires_grad = False                     #   True か False
model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)  # 2 最終層を NUM_CLASSES 個の出力に置き換える（新しい層は学習する）

<details><summary>答え（考えてから開く）</summary>

```python
NUM_CLASSES = 3
for param in model.parameters():                   # 1 全部を凍結する（読んだ直後は全部が学習する設定）
    param.requires_grad = False                      #   True か False
model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)  # 2 最終層を NUM_CLASSES 個の出力に置き換える（新しい層は学習する）
```

</details>

In [9]:
check_model(model, 3, "last")

OK｜出力 3 個｜学習するパラメータ 3,075 / 6,956,931（0.04%）


## 振り返り（このセルをダブルクリックして書く）

- 2 つのモデルの最終層の名前と入る数：ResNet18 fc・512／DenseNet121 classifier・1024
- モデルが変わっても同じだった所と，変えた所：全部を凍結してから最終層を nn.Linear(入る数, クラス数) に置き換える手順は同じ．変えたのは最終層の名前（fc と classifier）とクラス数だけ
- 最終層を置き換えてから「全部を凍結」すると，どうなるか：新しい最終層まで凍結されて，学習するパラメータが 0 になる．どの層も重みが変わらないので，何も学習できない

## 次：復習スライド（Google スライド 4 枚）

第5回の自分の `course1/day5/ex2.ipynb` の結果で，自分のモデルを説明する．中身は授業ページの「復習スライド」を見る．4 枚目には，この演習の図を参考に，選んだモデルで最終層の図を描く（最終層の出力は 2）．PDF にして `c1_d6_slides.pdf` で提出する．